In [1]:
from mxnet import np, npx
from mxnet.gluon import nn

npx.set_np()

net = nn.Sequential()
net.add(nn.Dense(256, activation='relu'))
net.add(nn.Dense(10))
net.initialize()

X = np.random.uniform(size=(2, 20))
net(X)

array([[ 0.06240273, -0.03268593,  0.02582652,  0.02254181, -0.03728798,
        -0.04253787,  0.00540613, -0.01364185, -0.09915451, -0.02272738],
       [ 0.02816679, -0.03341204,  0.03565667,  0.02506384, -0.04136415,
        -0.04941843,  0.0173853 ,  0.01081961, -0.09932576, -0.01176297]])

In [2]:
#自定义块
class MLP(nn.Block):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.hidden = nn.Dense(256, activation='relu')  # 隐藏层
        self.out = nn.Dense(10)  # 输出层
    def forward(self, X):
        return self.out(self.hidden(X))

In [3]:
net = MLP()
net.initialize()
net(X)

array([[-0.03989593, -0.10414708,  0.06799038,  0.05245075,  0.02526058,
        -0.00640342,  0.04182097, -0.01665319, -0.02067345, -0.07863817],
       [-0.03612847, -0.07210436,  0.09159478,  0.0789077 ,  0.02494172,
        -0.01028664,  0.01732428, -0.02843242,  0.03772651, -0.06671703]])

In [4]:
#顺序块
class MySequential(nn.Block):
    def add(self, block):
        self._children[block.name] = block

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._children.values():
            X = block(X)
        return X

In [5]:
net = MySequential()
net.add(nn.Dense(256, activation='relu'))
net.add(nn.Dense(10))
net.initialize()
net(X)

array([[-0.0764568 , -0.01130232,  0.04952145, -0.04651387, -0.04131571,
        -0.05884132, -0.06213812,  0.01311471, -0.01379424, -0.02514282],
       [-0.05124623,  0.00711232, -0.00155933, -0.07555379, -0.06675333,
        -0.01762914,  0.00589084,  0.0144719 , -0.04330775,  0.03317727]])

In [6]:
#forward（）中执行
class FixedHiddenMLP(nn.Block):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.rand_weight = self.params.get_constant(
            'rand_weight', np.random.uniform(size=(20, 20)))
        self.dense = nn.Dense(20, activation='relu')

    def forward(self, X):
        X = self.dense(X)
        X = npx.relu(np.dot(X, self.rand_weight.data()) + 1)
        X = self.dense(X)
        # 控制流
        while np.abs(X).sum() > 1:
            X /= 2
        return X.sum()

In [7]:
net = FixedHiddenMLP()
net.initialize()
net(X)

array(0.52637565)

In [8]:
#混搭
class NestMLP(nn.Block):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.net = nn.Sequential()
        self.net.add(nn.Dense(64, activation='relu'),
                     nn.Dense(32, activation='relu'))
        self.dense = nn.Dense(16, activation='relu')

    def forward(self, X):
        return self.dense(self.net(X))

chimera = nn.Sequential()
chimera.add(NestMLP(), nn.Dense(20), FixedHiddenMLP())
chimera.initialize()
chimera(X)

array(0.9772054)